In [13]:
import pandas as pd

In [25]:
df = pd.read_csv('sasrec_format.csv')
df_train = pd.read_csv('sasrec_format_by_user_train.csv')
df_test = pd.read_csv('sasrec_format_by_user_test.csv')

In [21]:
df.head()

,index,user_id,sequence_item_ids,sequence_ratings,sequence_timestamps,sex,age_group,occupation,zip_code
0,4775,4776,"457,1894,1367,2918,356,3157,36,628,3160,2501,3...","5,3,5,4,5,4,5,4,1,5,5,5,3,4,5,5,5,5,5,5,5,5,3,4,5","963028458,963028483,963028518,963028518,963028...",0,2,3,1534
1,3931,3932,"542,2403,3072,1278,1991,2462,3101,383,2401,137...","4,3,2,4,1,4,4,4,3,5,5,2,5,4,5,5,5,4,4,5,5,5,3,...","965703698,965703698,965703698,965703698,965703...",1,2,7,1681
2,5540,5541,"3256,1090,527,2312,3706,858,3470,2440,318,1221...","4,5,5,3,3,5,4,4,4,5,5,5,3,3,4,4,4,3,3,5,3,4,4,...","959522975,959522975,959522975,959522975,959523...",1,6,7,3026
3,1912,1913,"356,2135,1617,1357,912,750,3471,541,924,1214,1...","3,4,3,3,5,4,4,5,5,4,5,4,3,5,5,5,4,4,5,3,5,3,5,...","974691879,974691925,974691925,974691954,974691...",1,2,17,3175
4,2082,2083,"1347,1,1193,3572,2858,260,1196,2324,1219,593,3...","3,5,5,1,5,5,5,5,5,3,4,5,5,5,5,4,3,5,5,5,3,3,4,...","974654727,974654727,974654811,974654811,974654...",1,1,4,2187


In [17]:
df['sequence_ratings'].iloc[0]

'5,3,5,4,5,4,5,4,1,5,5,5,3,4,5,5,5,5,5,5,5,5,3,4,5'

In [26]:
# map sequence_ratings to binary values
def ratings_to_binary_threshold(rating_str, threshold=4):
    """将评分字符串转换为二进制值，>=threshold为1，<threshold为0"""
    ratings = [int(x) for x in rating_str.split(',')]
    return ','.join(['1' if rating >= threshold else '0' for rating in ratings])



In [27]:
df['sequence_ratings'] = df['sequence_ratings'].apply(lambda x: ratings_to_binary_threshold(x, 4))
df_train['sequence_ratings'] = df_train['sequence_ratings'].apply(lambda x: ratings_to_binary_threshold(x, 4))
df_test['sequence_ratings'] = df_test['sequence_ratings'].apply(lambda x: ratings_to_binary_threshold(x, 4))

In [23]:
df.head()

,index,user_id,sequence_item_ids,sequence_ratings,sequence_timestamps,sex,age_group,occupation,zip_code
0,4775,4776,"457,1894,1367,2918,356,3157,36,628,3160,2501,3...","1,0,1,1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1","963028458,963028483,963028518,963028518,963028...",0,2,3,1534
1,3931,3932,"542,2403,3072,1278,1991,2462,3101,383,2401,137...","1,0,0,1,0,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,0,...","965703698,965703698,965703698,965703698,965703...",1,2,7,1681
2,5540,5541,"3256,1090,527,2312,3706,858,3470,2440,318,1221...","1,1,1,0,0,1,1,1,1,1,1,1,0,0,1,1,1,0,0,1,0,1,1,...","959522975,959522975,959522975,959522975,959523...",1,6,7,3026
3,1912,1913,"356,2135,1617,1357,912,750,3471,541,924,1214,1...","0,1,0,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,0,1,...","974691879,974691925,974691925,974691954,974691...",1,2,17,3175
4,2082,2083,"1347,1,1193,3572,2858,260,1196,2324,1219,593,3...","0,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,0,0,1,...","974654727,974654727,974654811,974654811,974654...",1,1,4,2187


In [28]:
df.to_csv('sasrec_format_binary.csv', index=False)
df_train.to_csv('sasrec_format_by_user_train_binary.csv', index=False)
df_test.to_csv('sasrec_format_by_user_test_binary.csv', index=False)

In [18]:
# 方法1: 使用阈值转换 (推荐系统常用方法)
# 通常使用评分中位数或3/4作为阈值，这里使用4作为阈值
def ratings_to_binary_threshold(rating_str, threshold=4):
    """将评分字符串转换为二进制值，>=threshold为1，<threshold为0"""
    ratings = [int(x) for x in rating_str.split(',')]
    return ','.join(['1' if rating >= threshold else '0' for rating in ratings])

# 方法2: 使用中位数作为阈值
def ratings_to_binary_median(rating_str):
    """使用评分序列的中位数作为阈值"""
    ratings = [int(x) for x in rating_str.split(',')]
    median = sorted(ratings)[len(ratings)//2]
    return ','.join(['1' if rating >= median else '0' for rating in ratings])

# 方法3: 直接映射 1-2->0, 3-5->1
def ratings_to_binary_simple(rating_str):
    """简单映射：1-2分为0（不喜欢），3-5分为1（喜欢）"""
    mapping = {'1': '0', '2': '0', '3': '1', '4': '1', '5': '1'}
    return ','.join([mapping[x] for x in rating_str.split(',')])

# 测试示例
example_rating = '5,3,5,4,5,4,5,4,1,5,5,5,3,4,5,5,5,5,5,5,5,5,3,4,5'

print("原始评分:", example_rating)
print("方法1 (阈值=4):", ratings_to_binary_threshold(example_rating, 4))
print("方法2 (中位数):", ratings_to_binary_median(example_rating))
print("方法3 (简单映射):", ratings_to_binary_simple(example_rating))

# 应用到DataFrame列
# 选择您喜欢的方法，例如使用阈值=4的方法：
df['sequence_ratings_binary'] = df['sequence_ratings'].apply(lambda x: ratings_to_binary_threshold(x, 4))

# 查看转换结果
print("\n转换结果示例:")
print("原始:", df['sequence_ratings'].iloc[0])
print("二进制:", df['sequence_ratings_binary'].iloc[0])

原始评分: 5,3,5,4,5,4,5,4,1,5,5,5,3,4,5,5,5,5,5,5,5,5,3,4,5
方法1 (阈值=4): 1,0,1,1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1
方法2 (中位数): 1,0,1,0,1,0,1,0,0,1,1,1,0,0,1,1,1,1,1,1,1,1,0,0,1
方法3 (简单映射): 1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1

转换结果示例:
原始: 5,3,5,4,5,4,5,4,1,5,5,5,3,4,5,5,5,5,5,5,5,5,3,4,5
二进制: 1,0,1,1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1
